# Per-gene dose-response curve comparison

Generalized, dataset-agnostic version of `GEX_comp_Doming_Morris.ipynb` -- compares fitted bayesDREAM models across all overlapping trans genes, for however many datasets have a completed fit for a given cis gene. Panel shape auto-adapts: 2 datasets -> 2x2, 3 datasets -> 2x3.

Each panel:
- row 0: every dataset standalone (its own data + its own curve, with fitted-parameter markers)
- row 1: every dataset's own data + own curve again, with every *other* available dataset's curve overlaid on top

plus one guide-density panel (log2FC(x_true) by guide/cell_line) shared across all genes for a given cis gene.

There are two ways to generate these panels, both driven by the same underlying `save_model_for_plotting()` export per (dataset, cis_gene):

- **Heavy (`compare_datasets`/`make_panel`)**: reloads a full live model per dataset (raw counts + trans-fit posterior). Gives fitted-parameter markers (EC50/inflection) and a real guide-density panel, but memory scales with however many trans genes you ask for -- fine for Domingo's ~91-gene panel, not for holding Morris/Replogle's transcriptome-wide (~8-11k gene) panel in memory at once.
- **Lightweight (`compare_datasets_lightweight`/`make_panel_lightweight`)**: reads only each dataset's already-on-disk `trans_feature_summary_{modality}.csv` and a small precomputed smoothed-curve artifact -- no model, no raw counts, no trans-fit posterior. A few tens of MB even for Morris/Replogle's full panel, so this is what makes the notebook usable to plot **any** trans gene, from **any** dataset, on demand -- see the "Model-free" section below. Trade-off: no fitted-parameter markers, no guide-density panel (both need per-cell/per-sample data this path never loads).

**Prerequisite:** for each (dataset, cis_gene) you want here, `reconstruct_and_export()` (`comparative/reconstruct_export.py` for Domingo/Morris, `comparative/reconstruct_export_replogle.py` for Replogle) must already have been run once -- it writes both the `save_model_for_plotting()` export the heavy path needs and the precomputed smoothed-curve artifact the lightweight path needs, in one pass. The automation below just skips (with a printed message) any cis gene where a required export is still missing, rather than crash.

In [ ]:
# run "pip install ipython-autotime" in your conda env
%load_ext autotime

import os
import sys

# Derive the repo root from bayesDREAM's installed location (pip -e .), not
# from os.getcwd() -- the notebook's cwd at kernel start isn't guaranteed to
# be its own directory (depends on how Jupyter/the IDE was launched), so a
# '../..'-from-cwd guess silently fails to find comparative/ in that case.
# importlib.import_module (not a plain 'import bayesDREAM as ...') deliberately
# sidesteps a name collision: the bayesDREAM PACKAGE and the bayesDREAM CLASS it
# exports both share the literal name 'bayesDREAM'. If this cell's import ever
# gets merged with a 'from bayesDREAM import bayesDREAM' line elsewhere in the
# notebook, a plain import here can end up aliasing the class (no __file__)
# instead of the package -- this form can't be shadowed that way.
import importlib
_bayesdream_pkg = importlib.import_module('bayesDREAM')
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(_bayesdream_pkg.__file__)))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import matplotlib.pyplot as plt

from comparative.datasets import DOMINGO, MORRIS, REPLOGLE, morris_symbol_to_id
from comparative.dose_response_panels import (
    # Heavy path (full live model per dataset).
    compare_datasets,
    compare_all_domingo_cis_genes,
    load_model_for_plotting,
    make_panel,
    resolve_sum_factor_col,
    allsig_copy,
    # Lightweight path (summary CSV + precomputed smoothed curves only).
    compare_datasets_lightweight,
    make_panel_lightweight,
    load_gene_summary,
    load_smoothed_curves,
)

## Config

In [ ]:
deviceno = 2
DEVICE = f'cuda:{deviceno}' if torch.cuda.is_available() else 'cpu'

PLOT_DIR = './dose_response_comparison_plots'

# Standalone panels (row 0) show fitted-parameter markers (EC50/inflection
# lines) by default. Turn off if they read as confusing next to the
# dataset-colour curves -- overlay panels (row 1) never show markers
# regardless of this flag.
SHOW_PARAM_MARKERS = True

## Backfill precomputed exports (only runs what's missing)

The automated loop below (and the lightweight-load cells further down) read each dataset's `save_model_for_plotting()` export off disk -- including a precomputed smoothed-curve artifact (`smoothed_xy_{modality}.npz`). If an export predates that artifact being added, loading it raises `FileNotFoundError` telling you to backfill it.

This cell backfills every (dataset, cis gene) export, but only for genes that actually need it -- `reconstruct_and_export()`'s own checkpoint (`is_already_backfilled()`) skips anything already complete. Safe and cheap to re-run any time (e.g. at the top of a fresh kernel) -- already-complete exports come back near-instantly.

For Morris/Replogle, the smoothed-curve precompute itself is bounded to `domingo_union_features()` -- the UNION of Domingo's own trans-gene panels across ALL of its cis genes (GFI1B, NFE2, MYB, TET2), not their full ~10-20k-gene transcriptome-wide panel -- that full loop takes hours (confirmed ~2.1s/feature), most of it for genes never plotted by the automated comparison. This bound is deliberately cis-gene-independent (the SAME set for every cis gene of that dataset), so it still gives a meaningful precompute even for a cis gene Domingo never fit at all (HHEX/IKZF1/RUNX1) -- a per-cis-gene Domingo bound would have nothing to restrict to there. Any gene outside the bound is computed on the fly at plot time instead -- see the "Model-free" section further down.

In [ ]:
from comparative.reconstruct_export import reconstruct_and_export_all
from comparative.reconstruct_export_replogle import reconstruct_and_export_all as reconstruct_and_export_all_replogle

# force=False (default): each (dataset, cis gene) is skipped unless its
# summary CSV, save_model_for_plotting() export, or smoothed-curve artifact
# is missing/incomplete -- see reconstruct_export.py's is_already_backfilled().
# Only genes actually missing something get reconstructed (a full model
# reload, so real work for those -- possibly slow for Morris/Replogle's
# large panels). Pass force=True to a specific reconstruct_and_export(...)
# call instead if you need to redo one that's already complete (e.g. after
# re-fitting).
reconstruct_and_export_all()             # Domingo + Morris
reconstruct_and_export_all_replogle()    # Replogle

## Automated: every Domingo cis gene

The main entry point. Loops over every cis gene in `DOMINGO.cis_genes` (GFI1B, NFE2, MYB, TET2 -- Domingo bounds the trans gene panel size, so it's the dataset this loop is driven by), and for each one automatically uses whichever of {Domingo, Morris, Replogle} actually has a completed fit for that gene (checked via each `DatasetSpec.cis_genes`):

- **GFI1B, NFE2**: all 3 datasets have it -> 2x3 panel per trans gene
- **MYB, TET2**: Morris never fit these (see `publication_runs/morris/config.yaml`'s `primary_genes`) -> 2x2 Domingo-vs-Replogle panel per trans gene

No manual per-gene setup needed -- just run this cell. Writes into `PLOT_DIR/<cis_gene>/`.

In [ ]:
results = compare_all_domingo_cis_genes(
    out_dir=PLOT_DIR,
    datasets=[DOMINGO, MORRIS, REPLOGLE],
    lightweight=True,  # model-free path -- avoids OOM on Morris/Replogle's
                        # transcriptome-wide trans-fit posteriors. The heavy
                        # path (lightweight=False) has no way to bound memory
                        # here since it loads every trans gene's full
                        # posterior for every dataset; see the "Customising
                        # further" section below. show_param_markers/device
                        # are heavy-path-only kwargs, dropped accordingly.
)
print('\nSummary:')
for cis_gene, genes in results.items():
    print(f'  {cis_gene}: {len(genes)} trans genes plotted')

## One cis gene at a time

Useful while iterating on plot styling, or to force a specific dataset subset instead of auto-detecting from `cis_genes`.

In [ ]:
CIS_GENE = 'GFI1B'
SPECS = [DOMINGO, MORRIS, REPLOGLE]  # or e.g. [DOMINGO, REPLOGLE] to force just two

plotted = compare_datasets(
    SPECS, CIS_GENE,
    out_dir=os.path.join(PLOT_DIR, CIS_GENE),
    show_param_markers=SHOW_PARAM_MARKERS,
    device=DEVICE,
)
print(f'Plotted {len(plotted)} genes: {plotted[:10]}{"..." if len(plotted) > 10 else ""}')

## Inspect a single panel inline

Reload the models for `CIS_GENE` once, then call `make_panel` directly for one gene at a time -- faster than re-running the whole loop above while tweaking plot styling.

In [ ]:
models = [load_model_for_plotting(s, CIS_GENE, device=DEVICE) for s in SPECS]
sfcols = [resolve_sum_factor_col(s, m) for s, m in zip(SPECS, models)]
summaries = [m.save_trans_summary(compute_lfc_ci=False, compute_derivative_roots=False) for m in models]
summaries_allsig = [allsig_copy(s, spec) for s, spec in zip(summaries, SPECS)]

In [ ]:
# plotted[0] (from cell 7) is a display SYMBOL -- make_panel() itself now
# expects the canonical gene_id (Ensembl), translated the same way
# compare_datasets() does internally (see allsig_copy()'s docstring).
GOI_symbol = plotted[0] if plotted else None
GOI = morris_symbol_to_id().get(GOI_symbol, GOI_symbol) if GOI_symbol else None
fig, unified_x = make_panel(
    GOI, SPECS, models, summaries_allsig, sfcols,
    cis_gene=CIS_GENE, show_param_markers=SHOW_PARAM_MARKERS, display_name=GOI_symbol,
)
plt.show()

## Model-free: plot any trans gene on demand (lightweight)

The lightweight path never loads a model, raw counts, or a trans-fit posterior -- just each dataset's `trans_feature_summary_{modality}.csv` (a few hundred columns) and a small precomputed smoothed-curve array. Both are cheap enough to hold for a dataset's **entire** trans panel at once, so once loaded below, any gene can be plotted instantly -- no per-gene reload, unlike the "Inspect a single panel inline" section above (which still constructs a full model per dataset).

For Morris/Replogle, the precomputed smoothed-curve artifact only covers the genes in `comparative.dose_response_panels.domingo_union_features()` -- the union of Domingo's own trans-gene panels across all its cis genes -- not their full ~10-20k-gene transcriptome-wide panel; that full loop takes hours (confirmed ~2.1s/feature), not worth paying up front for genes that are rarely, if ever, plotted. Asking for a gene outside that set below (or for a cis gene Domingo never fit at all, e.g. HHEX/IKZF1/RUNX1 -- the union bound still applies there, same as for GFI1B/NFE2/MYB/TET2) is still fine -- `plot_gene_lightweight()`/`make_panel_lightweight()` compute it on the fly (`ensure_smoothed_curve()`), reloading a lean, trans-posterior-free model just for that gene, print a note that they're doing so, and cache the result (in memory, and by default persisted back to disk) so asking for the same gene again is instant.

In [ ]:
# Load once per dataset -- cheap (a summary CSV + a small .npz), holds the
# ENTIRE trans panel, not just CIS_GENE's shared subset. Re-run this cell
# whenever you change CIS_GENE.
summary_pairs = {s.name: load_gene_summary(s, CIS_GENE) for s in SPECS}
lw_summaries = {name: p[0] for name, p in summary_pairs.items()}
lw_summaries_allsig = {name: p[1] for name, p in summary_pairs.items()}
lw_smoothed = {s.name: load_smoothed_curves(s, CIS_GENE) for s in SPECS}
print(f"Loaded {CIS_GENE}: " + ", ".join(
    f"{name} ({len(df)} trans genes)" for name, df in lw_summaries.items()
))

In [ ]:
# Plot ANY trans gene, instantly -- no reload. Change GENE_SYMBOL and re-run
# this cell as many times as you like; the load-once cell above never needs
# to run again unless CIS_GENE changes. Accepts a symbol or an Ensembl ID
# (mixed across datasets is fine, same as compare_datasets(genes=[...])).
GENE_SYMBOL = 'MYC'  # <- edit this

goi = morris_symbol_to_id().get(GENE_SYMBOL, GENE_SYMBOL)
fig, unified_x = make_panel_lightweight(
    goi, SPECS,
    [lw_summaries[s.name] for s in SPECS],
    [lw_summaries_allsig[s.name] for s in SPECS],
    [lw_smoothed[s.name] for s in SPECS],
    cis_gene=CIS_GENE, display_name=GENE_SYMBOL,
)
plt.show()

## Customising further

- `compare_all_domingo_cis_genes(bounding_dataset=REPLOGLE)` would instead loop over Replogle's (larger) cis-gene list.
- `compare_all_domingo_cis_genes(lightweight=True)` (and `compare_datasets(...)` -> `compare_datasets_lightweight(...)`) switches the automated/one-cis-gene-at-a-time loops to the model-free path -- same file outputs, but scales to a dataset's full trans panel instead of only Domingo's ~91-gene shared set (see the "Model-free" section above for the interactive equivalent). No `show_param_markers` or guide-density panel in that mode -- both need per-cell/per-sample data the lightweight path never loads.
- Pass `genes=[...]` to `compare_datasets`/`compare_datasets_lightweight`/`compare_all_domingo_cis_genes` to restrict to a short hand-picked trans gene list instead of every shared one.
- Genes are matched across datasets by Ensembl gene_id, not symbol -- Domingo/Morris's own feature identity is the symbol, Replogle's is the Ensembl ID; `comparative.datasets.morris_symbol_to_id()`/`morris_id_to_symbol()` (built from Morris's transcriptome-wide gene_meta.csv) bridge the two. A `genes=[...]` list can be symbols or gene_ids, mixed.
- **Memory (heavy path)**: `lean=True` (default) collapses the NTC/cis posteriors to point estimates but never touches the trans-fit posterior -- `load_trans_fit(lean=True)` is not implemented in bayesDREAM (raises `NotImplementedError`; nearly every `save_trans_summary()` column needs the full joint per-draw posterior, not just its marginal). The trans-fit posterior is the dominant memory cost for Morris/Replogle's transcriptome-wide panels, and `lean` alone does not shrink it -- for that, use the lightweight path instead (above), which never loads a trans-fit posterior at all.
- Each `DatasetSpec` in `comparative/datasets.py` carries its own fit-curve `.color` and `.cell_line_palette` -- edit there, not here, to keep both notebooks in sync.
- Row-1's "own curve" (both paths) is force-rendered regardless of that dataset's own FDR significance (matching the original 2-way panels' intent: show the shape even where it isn't a formal call); row-0's standalone curve uses real per-dataset significance gating, so a blank row-0 curve for a given dataset is real signal, not a bug.